# PyTorch Experience · 05
## `torch.no_grad()` vs `torch.inference_mode()`

**One glance:** Both skip recording the backward graph. **`inference_mode()` reduces more bookkeeping**, but its tensors have extra restrictions.

[← Pytorch Experience](https://lawson-dong.vercel.app/coding/pytorch-experience) · [PyTorch Autograd modes](https://docs.pytorch.org/docs/stable/notes/autograd.html#grad-modes)

### 1 · See the difference

![Comparison of normal autograd, no_grad and inference_mode](https://raw.githubusercontent.com/Lawson-Dong/personal-website/main/public/images/pytorch-gradient-modes.svg)

| | `no_grad()` | `inference_mode()` |
|---|---|---|
| Record backward graph? | No | No |
| Extra tensor bookkeeping? | Some remains | Less |
| Reuse new tensors in future autograd calculations? | Usually yes | Restricted |
| Good for | Updating parameters or values without tracking | Pure inference |

Both are **context managers**. They do **not** replace `model.eval()` or run `model(x)` automatically.

### 2 · Experiment 1: Three ways to run forward

A single Linear layer is enough. This runs on CPU and needs no downloads.

In [ ]:
import torch
from torch import nn

model = nn.Linear(4, 2)
x = torch.randn(3, 4)

y1 = model(x)
with torch.no_grad():
    y2 = model(x)
with torch.inference_mode():
    y3 = model(x)

print('Normal requires_grad:   ', y1.requires_grad)  # True
print('no_grad requires_grad:  ', y2.requires_grad)  # False
print('inference requires_grad:', y3.requires_grad)  # False
print('Same values:', torch.allclose(y1, y2) and torch.allclose(y2, y3))

### 3 · Experiment 2: Can you use that tensor in a later backward pass?

**Important:** `requires_grad=False` does not tell the whole story. Tensors made under inference mode carry additional restrictions.

Here, multiplying a trainable parameter by a **normal tensor** is allowed, but multiplying it by an **inference tensor** raises a RuntimeError when autograd needs to save the tensor for backward.

In [ ]:
weight = nn.Parameter(torch.tensor([1., 1., 1.]))

with torch.no_grad():
    ordinary = torch.tensor([2., 3., 4.])

with torch.inference_mode():
    inference_tensor = torch.tensor([2., 3., 4.])

# Allowed: ordinary is a normal tensor
(weight * ordinary).sum().backward()
print('Gradient from no_grad tensor:', weight.grad)

try:
    (weight * inference_tensor).sum().backward()
except RuntimeError as error:
    print('Inference tensor cannot be used here:', str(error).split('.')[0])

# Create a normal copy OUTSIDE inference mode if necessary
normal_copy = inference_tensor.clone()
weight.grad = None
(weight * normal_copy).sum().backward()
print('Gradient after clone:', weight.grad)

### 4 · Common inference template

```python
model.eval()  # Evaluation behavior: e.g. Dropout, BatchNorm
with torch.inference_mode():
    output = model(x)  # Forward pass, without autograd tracking
```

- `model.eval()` controls the model's **training/evaluation behavior**.
- `torch.no_grad()` and `torch.inference_mode()` control **gradient tracking**.
- `torch.inference_mode()` can reduce overhead, but it is **not guaranteed to speed up every workload**.

**Official docs:** [torch.no_grad()](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html) · [torch.inference_mode()](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html) · [Autograd Grad Modes](https://docs.pytorch.org/docs/stable/notes/autograd.html#grad-modes)